# wikisource-extract

[German Wikisource](https://de.wikisource.org/wiki/Heliand) reproduces [Behaghel's edition](https://archive.org/details/heliandundgenesi00beha/) of the _Heliand_ (using M, i.e. [Munich, Bayerische Staatsbibliothek, Cgm 25](https://www.digitale-sammlungen.de/de/view/bsb00026305), as its base manuscript, though with heavy emendation). The edition used is presumably the fourth, from 1922, as that's the public domain edition of record. The encoder has added puncti to represent caesuras, and the text has not been proofread, witness the form "atquepraeclaro" in the preface.

This notebook strips the HTML file of its tags and outputs a clean text to JSON and plaintext.

In [1]:
import json,re
from mediawiki import MediaWiki
from pathlib import Path
from bs4 import BeautifulSoup

In [2]:
print_line_numbers = True
caesura_span = '    '
#caesura_span = '\t'

normalization = {
    'â': 'a',
    'ê': 'e',
    'î': 'i',
    'ô': 'o',
    'û': 'u',
    '[': '',
    ']': '',
    'ʽ': '',
    'ʼ': '',
    '.': '',
    ',': '',
    ':': '',
    ';': '',
    '—': '',
    '!': '',
    '?': '',
    '*': ''
}

def normalize(token):
    token = token.lower()
    for k,v in normalization.items():
        token = token.replace(k,v)
    return token

In [3]:
wikisource = MediaWiki(url='https://secure.wikimedia.org/wikisource/de/w/api.php')
page = wikisource.page('Heliand')
html = page.html
soup = BeautifulSoup(html, 'html.parser')
poem = soup.find('div', class_='poem')
plaintext = poem.get_text().split('\n')[1:]

In [4]:
heliand = []
caesura = ' · '
previous_line = ''
for line in plaintext:
    if re.search(r'\d', line):
        if caesura in line:
            line = (line.split(' ', 1)[0], line.split(' ', 1)[1].split(caesura))
            if not(re.search(r'(\w|.)', line[1][0])):
                data = (heliand[-1][0], previous_line[1].split(), line[1][1].split())
                heliand.pop()
            else:
                data =(line[0], line[1][0].split(), line[1][1].rstrip().split())
        else:
            line = (line.split(' ', 1)[0], line.split(' ', 1)[1])
            data = (line[0], line[1][0].split())
        heliand.append(data)
    previous_line = line

heliand_clean = []
for line in heliand:
    line_no = line[0].replace('b', 'x')
    on_verse = {
        'verse': line_no + 'a',
        'tokens': [normalize(i) for i in line[1]]
    }
    off_verse = {
        'verse': line_no + 'b',
        'tokens': [normalize(i) for i in line[2]]
    }
    heliand_clean.append(on_verse)
    heliand_clean.append(off_verse)


In [5]:
json_file_clean = 'heliand-m.json'
if not(Path(json_file_clean).is_file()):
    print('Generating heliand-m.json...')
    with open(json_file_clean, 'w', encoding='utf-8') as outfile:
        json.dump(heliand_clean, outfile, ensure_ascii=False, indent=4)

json_file_edited = 'heliand-m_behaghel.json'
if not(Path(json_file_edited).is_file()):
    print('Generating heliand-m_behaghel.json...')
    with open(json_file_edited, 'w', encoding='utf-8') as outfile:
        json.dump(heliand, outfile, ensure_ascii=False, indent=4)

Generating heliand-m.json...
Generating heliand-m_behaghel.json...


In [6]:
plaintext_file_edited = 'heliand-m_behaghel.txt'
if Path(plaintext_file_edited).is_file():
    print('heliand-m_behaghel.txt already in place. Skipping.')
else:
    print('Generating heliand-m_behaghel.txt...')
    verse_lines = []
    for line in heliand:
        if print_line_numbers == True:
            if 'b' in line[0]:
                zusatz = 'b'
            else:
                zusatz = ' '
            line_no = str("{:04d}".format(int(line[0].rstrip('b')))) + zusatz
            reconstructed_line = line_no + ' ' + ' '.join(line[1]) + caesura_span + ' '.join(line[2])
        else:
            reconstructed_line = ' '.join(line[1]) + caesura_span + ' '.join(line[2])
        verse_lines.append(reconstructed_line)
    with open(plaintext_file_edited, 'w') as outfile:
        outfile.write('\n'.join(verse_lines))

plaintext_file_clean = 'heliand-m.txt'
if Path(plaintext_file_clean).is_file():
    print('heliand-m.txt already in place. Skipping.')
else:
    print('Generating heliand-m.txt...')
    verse_lines = []
    for verse in heliand_clean:
        if len(verse['tokens']) == 1 and len(verse['tokens'][0]) < 1:
            empty = '                   '
        else:
            empty = ''
        tokens = [token for token in verse['tokens'] if len(token) > 0]
        if 'a' in verse['verse']:
            if print_line_numbers == True:
                if 'x' in verse['verse']:
                    zusatz = 'x'
                else:
                    zusatz = ' '
                line_no = str("{:04d}".format(int(verse['verse'].rstrip('ax')))) + zusatz + ' '
            else:
                line_no = ''
            reconstructed_line = line_no + empty + ' '.join(tokens)
            verse_lines.append(reconstructed_line)
        else:
            verse_lines[-1] = verse_lines[-1] + caesura_span + ' '.join(tokens)
            
    with open(plaintext_file_clean, 'w') as outfile:
        outfile.write('\n'.join(verse_lines))



Generating heliand-m_behaghel.txt...
Generating heliand-m.txt...
